In [2]:
import os
import mlflow
import pandas as pd

mlflow.set_tracking_uri("https://mlflow.smbgarasibmw.my.id")

base_dir = "hasil_evaluasi"
excel_dir = os.path.join(base_dir, "excel")
csv_dir = os.path.join(base_dir, "csv")

os.makedirs(excel_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

target_experiments = [
    "Phase_1_w2v_Text_Representation_perfold_audit",
    "Phase_1_SBERT_Text_Representation_PerFold_Au",  # otomatis fallback jika di server tertulis '...PerFold_Audit'
    "Phase_1_TFIDF_Text_Representation_PerFold_Audit",
]

for exp_name in target_experiments:
  exp = mlflow.get_experiment_by_name(exp_name)

  if exp is None and exp_name.endswith("_Au"):
    exp = mlflow.get_experiment_by_name(exp_name + "dit")
    if exp is not None:
      exp_name = exp_name + "dit"

  if exp is None:
    print(f"[SKIP] Eksperimen '{exp_name}' tidak ditemukan di server.")
    continue

  print(f"[PROSES] Mengambil data: {exp_name} (ID: {exp.experiment_id})...")

  df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
  if df.empty:
    print(f" -> Kosong, tidak ada run di '{exp_name}'.")
    continue

  metric_cols = [c for c in df.columns if c.startswith("metrics.")]
  info_cols = ["tags.mlflow.runName", "status", "start_time"]
  selected_cols = [c for c in info_cols if c in df.columns] + metric_cols

  export_df = df[selected_cols].copy()
  export_df.rename(columns=lambda x: x.replace("metrics.", ""), inplace=True)

  csv_path = os.path.join(csv_dir, f"{exp_name}.csv")
  export_df.to_csv(csv_path, index=False)
  print(f" -> [CSV] Tersimpan: {csv_path}")

  for col in export_df.select_dtypes(include=["datetimetz"]).columns:
    export_df[col] = export_df[col].dt.tz_localize(None)

  excel_path = os.path.join(excel_dir, f"{exp_name}.xlsx")
  export_df.to_excel(excel_path, index=False)
  print(f" -> [Excel] Tersimpan: {excel_path}")

print(
    f"\nSelesai! Seluruh file Excel dan CSV tersimpan di dalam folder"
    f" '{base_dir}/'."
)

[PROSES] Mengambil data: Phase_1_w2v_Text_Representation_perfold_audit (ID: 11)...
 -> [CSV] Tersimpan: hasil_evaluasi\csv\Phase_1_w2v_Text_Representation_perfold_audit.csv
 -> [Excel] Tersimpan: hasil_evaluasi\excel\Phase_1_w2v_Text_Representation_perfold_audit.xlsx
[PROSES] Mengambil data: Phase_1_SBERT_Text_Representation_PerFold_Au (ID: 10)...
 -> [CSV] Tersimpan: hasil_evaluasi\csv\Phase_1_SBERT_Text_Representation_PerFold_Au.csv
 -> [Excel] Tersimpan: hasil_evaluasi\excel\Phase_1_SBERT_Text_Representation_PerFold_Au.xlsx
[PROSES] Mengambil data: Phase_1_TFIDF_Text_Representation_PerFold_Audit (ID: 9)...
 -> [CSV] Tersimpan: hasil_evaluasi\csv\Phase_1_TFIDF_Text_Representation_PerFold_Audit.csv
 -> [Excel] Tersimpan: hasil_evaluasi\excel\Phase_1_TFIDF_Text_Representation_PerFold_Audit.xlsx

Selesai! Seluruh file Excel dan CSV tersimpan di dalam folder 'hasil_evaluasi/'.
